# 02 — Pré-processamento

Limpeza, tratamento de ausentes, encoding e normalização/padronização.

In [1]:
# Imports
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
%load_ext autoreload
%autoreload 2
from utils import load_parquet_safe

df_path = "../data/df_model_raw.parquet"

df = load_parquet_safe(df_path, '01_eda.ipynb')

print("Dados carregados com sucesso!")

Dados carregados com sucesso!


In [3]:
df.shape

(717932, 22)

In [4]:
df

,SEMAGESTAC,GESTACAO,IDADEMAE,ESCMAE2010,ESTCIVMAE,RACACORMAE,QTDGESTANT,QTDPARTNOR,QTDPARTCES,QTDFILVIVO,QTDFILMORT,GRAVIDEZ,MESPRENAT,CONSPRENAT,SEXO,KOTELCHUCK,IDADEPAI,LATITUDE,LONGITUDE,PREMATURO,PAI_AUSENTE,IDADEPAI_INVALIDA
0,38.0,5.0,30,5.0,1.0,1,1.0,0.0,1.0,1.0,0.0,1.0,1.0,11.0,1,5,39.0,-19.9102,-43.9266,0,0,0
1,38.0,5.0,27,3.0,1.0,1,0.0,0.0,0.0,0.0,0.0,1.0,2.0,8.0,2,5,39.0,-19.9102,-43.9266,0,0,0
2,40.0,5.0,41,5.0,1.0,4,0.0,0.0,0.0,0.0,0.0,1.0,1.0,9.0,2,5,36.0,-19.9102,-43.9266,0,0,0
3,37.0,5.0,29,3.0,1.0,1,0.0,0.0,0.0,0.0,0.0,1.0,1.0,9.0,2,5,22.0,-19.9102,-43.9266,0,0,0
4,40.0,5.0,36,5.0,4.0,1,2.0,0.0,0.0,0.0,2.0,1.0,1.0,9.0,2,5,38.0,-19.9102,-43.9266,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
717927,37.0,5.0,37,3.0,2.0,4,2.0,0.0,1.0,1.0,1.0,1.0,2.0,11.0,1,5,38.0,-21.3009,-46.7964,0,0,0
717928,39.0,5.0,37,3.0,2.0,4,1.0,0.0,1.0,1.0,0.0,1.0,2.0,8.0,1,5,31.0,-21.3050,-46.7081,0,0,0
717929,40.0,5.0,32,2.0,1.0,,1.0,0.0,1.0,1.0,0.0,1.0,1.0,12.0,2,5,22.0,-21.3590,-46.9401,0,0,0
717930,41.0,5.0,15,3.0,1.0,,0.0,0.0,0.0,0.0,0.0,1.0,1.0,7.0,2,5,19.0,-21.3050,-46.7081,0,0,0


## Convertendo Sentinelas (Sentinelas -> NaNs)
O SINASC tem sentinelas que são ignorados pelo sistema, porém contentdo 9,99,999 como códigos a serem ignorados.

Objetivo: Equalizar os dados para que o Pandas consiga comparar as linhas de forma justa na etapa de duplicadas

### Como identificar as colunas que tem sentinela?

#### 1. Técnica da Frequência de Finais
Ao fazer um `value_counts().head(10)`conseguimos o Top 10 valores mais populares por coluna

In [5]:
cols_to_check = df.columns

for col in cols_to_check:
    top_values = df[col].value_counts(normalize=True).head(10)
    print(f"Coluna: {col}")
    print(top_values)
    print("-" * 30)

Coluna: SEMAGESTAC
SEMAGESTAC
39.0    0.292670
38.0    0.231438
40.0    0.173330
37.0    0.109494
41.0    0.063487
36.0    0.043616
35.0    0.022797
34.0    0.015442
42.0    0.010665
33.0    0.008632
Name: proportion, dtype: float64
------------------------------
Coluna: GESTACAO
GESTACAO
5.0    0.870418
4.0    0.096395
6.0    0.016582
3.0    0.010546
2.0    0.005475
1.0    0.000584
Name: proportion, dtype: float64
------------------------------
Coluna: IDADEMAE
IDADEMAE
26    0.050069
25    0.049782
27    0.049726
28    0.049204
30    0.048591
29    0.048590
24    0.048392
31    0.047537
23    0.046957
22    0.046346
Name: proportion, dtype: float64
------------------------------
Coluna: ESCMAE2010
ESCMAE2010
3.0    0.557242
5.0    0.202487
2.0    0.167648
4.0    0.049514
1.0    0.020321
9.0    0.001443
0.0    0.001345
Name: proportion, dtype: float64
------------------------------
Coluna: ESTCIVMAE
ESTCIVMAE
1.0    0.452909
2.0    0.419698
5.0    0.101127
4.0    0.021165
9.0    0.002

### Como identificar o que é "lixo" do que é dado real? Será que todos os 9 na tabela toda são códigos de "ignorado"?

#### 2. Três critérios: Frequência, semântica e descontinuidade

Às vezes pelo nome da tabela essa informação consegue ser óbvia, mas outra vezes conseguimos seguir pelo o que os dados estão dizendo.

1. Onde o 9 e 99 são claramente SENTINELAS (lixo)
    - `MESPRENAT` **(99.0 com 2%)**: É sentinela. Ninguém começa o pré-natal no 99º mês. Note que o 9 aqui não aparece, o erro padrão do SINASC para meses é 99.
    - `ESTCIVMAE` e `ESCMAE2010` **(9.0 com ~0.2%)**: São sentinelas. Estão no final da lista, com frequências baixas, e o dicionário do SINASC reserva o 9 para "Ignorado" em variáveis categóricas codificadas de 1 a 5.
    - `SEXO` **(0 com 0.01%)**: O 0 aqui é sentinela para "Ignorado".

2. Onde o 9 é um VALOR REAL (dado válido)
    - Nestas colunas, o 9 faz parte de uma distribuição que vai diminuindo organicamente.
        - `QTDGESTANT, QTDPARTNOR, QTDFILVIVO`: * Observe a sequência: ...7, 8, 9, 10, 11...
            - O valor 9.0 aparece com uma frequência de **0.000650** (muito pequena). Se fosse um erro do sistema para "ignorado", esse número seria muito mais alto (como os 2% do `MESPRENAT`).
            - Aqui, 9 significa que a mãe realmente teve 9 gestações/partos. É um dado raro, mas biologicamente possível e importante para o modelo (multiparidade é fator de risco).
     
3. O caso especial: `RACACORMAE`
    - Note que em `RACACORMAE` apareceu um valor vazio com **2.5%**.
        - Isso é um "missing" disfarçado de string vazia. Você deve converter isso para NaN antes de qualquer outra coisa.


In [6]:
# Cria uma cópia para limpeza
df_clean = df.copy()

# 1. Padronização de tipos para fazer o mapeamento das sentinelas pois o .replace tem diferença entre types
cols_to_numeric = ['ESTCIVMAE', 'KOTELCHUCK', 'ESCMAE2010', 'GRAVIDEZ', 'SEXO', 'MESPRENAT', 'CONSPRENAT']

for col in cols_to_numeric:
    if col in df_clean.columns:
        # errors='coerce' limpa strings vazias ou caracteres estranhos
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# 2. Mapeamento cirúrgico de sentinelas
sentinels_mapping = {
    'ESTCIVMAE': [9, 9.0],
    'KOTELCHUCK': [9, 9.0],
    'ESCMAE2010': [9, 9.0],
    'GRAVIDEZ': [9, 9.0],
    'SEXO': [0, 9, 9.0],
    'MESPRENAT': [99, 99.0],
    'CONSPRENAT': [99, 99.0]
}

for col, sentinels in sentinels_mapping.items():
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].replace(sentinels, np.nan)


## Limpeza de Strings e Tipagem
Algumas strings na base que podem conter sugeira, como por exemplo `"MG"` tendo valores como `"MG "`(contendoespaços) o que é considerado diferente para o computador.

Objetivo: Remover espaços em branco e garantir que as colunas numéricas sejam de fato `float` ou `int`.

##### ps: a parte de tipagem já foi tratada no 01_eda.ipynb

In [7]:
# Tratamento de string vazia (ex: RACACORMAE)
df_clean = df_clean.replace(r'^\s*$', np.nan, regex=True)

## Identificação e Remoção de Duplicadas

Após a "normalização" dos dados (sentinelas viraram nulos e espaços foram removidos) conseguimos uma base melhor tratada para identificar duplicadas e removê-las.

Objetivo: Identificar e remover as duplicadas

In [8]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)

print(f"Registros removidos: {before - after}")

Registros removidos: 14782


## Identificação de Nulos

Agora que removemos as duplicadas precisamos tratar os nulos, isto é, nosso modelo não conseguiria processar valores não numéricos.

In [9]:
# 1. Ver contagem absoluta e percentual de nulos por coluna
null_summary = pd.DataFrame({
    'Nulos': df_clean.isnull().sum(),
    'Percentual (%)': (df_clean.isnull().mean() * 100).round(2)
})

# 2. Filtrar apenas as colunas que possuem algum nulo
print(null_summary[null_summary['Nulos'] > 0].sort_values(by='Nulos', ascending=False))

             Nulos  Percentual (%)
IDADEPAI    384422           54.67
MESPRENAT    24253            3.45
KOTELCHUCK   23919            3.40
RACACORMAE   16637            2.37
QTDPARTCES   10364            1.47
QTDPARTNOR   10242            1.46
QTDFILMORT   10022            1.43
QTDGESTANT    8712            1.24
QTDFILVIVO    7543            1.07
CONSPRENAT    7471            1.06
ESCMAE2010    5639            0.80
ESTCIVMAE     5412            0.77
GRAVIDEZ       288            0.04
SEXO            84            0.01
LATITUDE         7            0.00
LONGITUDE        7            0.00


## Feature Engineering

#### Mas porquê fazer a feature engineering _antes_ da imputação?
1. Preservação do Sinal (O caso do `PAI_AUSENTE`)
        Se você tem 50% de nulos na `IDADEPAI`, isso provavelmente indica que o pai não foi declarado no registro (de acordo com a realidade estatística de ausência paterna no Brasi).
        - **Se você imputar primeiro**: O nulo vira 30 (mediana). Quando você for criar a feature `PAI_AUSENTE`, não haverá mais nulos para identificar. O modelo verá um pai de 30 anos "comum".
        - **Se você fizer a engenharia antes**: Você marca `PAI_AUSENTE = 1`. Depois, você preenche a idade com `30`. O modelo agora tem duas informações: "Esse pai não está presente" **E** "Para fins de cálculo, considere a idade média de 30".

2. Evitar o "Ruído" em Regras de Negócio
        Algumas variáveis que criamos são baseadas em limites (ex: `PN_TARDIO` para quem começou o pré-natal após o 3º mês).
        - Se a mediana de `MESPRENAT` for `2` e você preencher os nulos antes, todas as mulheres que não informaram o mês de início serão classificadas como "Pré-natal Precoce" (mês 2).
        - Ao fazer a engenharia antes, você pode decidir se o nulo deve ser uma categoria à parte ou se deve ser tratado como risco, sem deixar que a mediana tome essa decisão por você.

3. Evitar Dados "Alucinados" (O caso da `PRIMIPARA`)
    Para saber se a mãe é de primeira viagem, olhamos se `QTDFILVIVO + QTDFILMORT == 0`.
    - Se você imputar a mediana em uma dessas colunas antes (digamos que a mediana de filhos vivos seja `1`), uma mãe que era nula (e possivelmente primípara) passará a ter `1` filho no cálculo. Você "inventou" um filho para ela e agora a feature `PRIMIPARA` será `0` (falso), quando poderia ser `1` (verdadeiro).


Se você inverte a ordem, você dá ao algoritmo dados "médios" e retira dele a capacidade de entender as exceções e os riscos sociais que o dado faltante representa.


#### Por que criar Ranges (Bins)?
Mães muito jovens ou em idade avançada possuem riscos biológicos e sociais completamente diferentes. Criar "faixas" ajuda o modelo a captar esses saltos de risco que um número contínuo (15, 16, 17...) às vezes mascara.

Aqui estão os ranges sugeridos baseados em literatura médica e nos padrões do SUS:

- **Extremo Baixa Idade (< 15 anos)**: Risco altíssimo, geralmente associado a vulnerabilidade social e imaturidade biológica.
- **Adolescente (15-19 anos)** Risco moderado/alto.
- **Idade Ideal (20-34 anos)**: Faixa de menor risco biológico (nossa base de comparação).
- **Idade Materna Avançada (35-44 anos)**: Risco crescente de hipertensão, diabetes gestacional e prematuridade.
- **Extremo Alta Idade (45+ anos)**: Risco muito elevado.

In [10]:
# Pré-natal Tardio (Inicio após o 3° mês)
df_clean['PNTARDIO'] = np.where(
    df_clean["MESPRENAT"].isna(), -1, (df_clean["MESPRENAT"] > 3).astype(int)
)

# Histórico de Perda (Se já teve algum filho morto)
df_clean['HISTPERDAFETAL'] = np.where(
    df_clean["QTDFILMORT"].isna(), -1, (df_clean['QTDFILMORT'] > 0).astype(int)
 )

# Primipara (Mãe na primeira gestação)
hist_children_absent = df_clean[["QTDFILVIVO", "QTDFILMORT"]].isna().any(axis=1)
total_children = df_clean[["QTDFILVIVO", "QTDFILMORT"]].sum(axis=1)

df_clean['PRIMIPARA'] = np.where(
    hist_children_absent, -1, (total_children == 0).astype(int)
)

# Categorização de idade
bins = [0, 14, 19, 34, 44 ,100]
labels = ['EXTBAIXA', 'ADOLESC', 'IDEAL', 'AVANCADA', 'EXTRALTA']

df_clean['FAIXAETAMAE'] = pd.cut(df_clean['IDADEMAE'], bins=bins, labels=labels)

df_clean


,SEMAGESTAC,GESTACAO,IDADEMAE,ESCMAE2010,ESTCIVMAE,RACACORMAE,QTDGESTANT,QTDPARTNOR,QTDPARTCES,QTDFILVIVO,QTDFILMORT,GRAVIDEZ,MESPRENAT,CONSPRENAT,SEXO,KOTELCHUCK,IDADEPAI,LATITUDE,LONGITUDE,PREMATURO,PAI_AUSENTE,IDADEPAI_INVALIDA,PNTARDIO,HISTPERDAFETAL,PRIMIPARA,FAIXAETAMAE
0,38.0,5.0,30,5.0,1.0,1,1.0,0.0,1.0,1.0,0.0,1.0,1.0,11.0,1.0,5.0,39.0,-19.9102,-43.9266,0,0,0,0,0,0,IDEAL
1,38.0,5.0,27,3.0,1.0,1,0.0,0.0,0.0,0.0,0.0,1.0,2.0,8.0,2.0,5.0,39.0,-19.9102,-43.9266,0,0,0,0,0,1,IDEAL
2,40.0,5.0,41,5.0,1.0,4,0.0,0.0,0.0,0.0,0.0,1.0,1.0,9.0,2.0,5.0,36.0,-19.9102,-43.9266,0,0,0,0,0,1,AVANCADA
3,37.0,5.0,29,3.0,1.0,1,0.0,0.0,0.0,0.0,0.0,1.0,1.0,9.0,2.0,5.0,22.0,-19.9102,-43.9266,0,0,0,0,0,1,IDEAL
4,40.0,5.0,36,5.0,4.0,1,2.0,0.0,0.0,0.0,2.0,1.0,1.0,9.0,2.0,5.0,38.0,-19.9102,-43.9266,0,0,0,0,1,0,AVANCADA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
717927,37.0,5.0,37,3.0,2.0,4,2.0,0.0,1.0,1.0,1.0,1.0,2.0,11.0,1.0,5.0,38.0,-21.3009,-46.7964,0,0,0,0,1,0,AVANCADA
717928,39.0,5.0,37,3.0,2.0,4,1.0,0.0,1.0,1.0,0.0,1.0,2.0,8.0,1.0,5.0,31.0,-21.3050,-46.7081,0,0,0,0,0,0,AVANCADA
717929,40.0,5.0,32,2.0,1.0,NaN,1.0,0.0,1.0,1.0,0.0,1.0,1.0,12.0,2.0,5.0,22.0,-21.3590,-46.9401,0,0,0,0,0,0,IDEAL
717930,41.0,5.0,15,3.0,1.0,NaN,0.0,0.0,0.0,0.0,0.0,1.0,1.0,7.0,2.0,5.0,19.0,-21.3050,-46.7081,0,0,0,0,0,1,ADOLESC


## Imputação - Tratamento de Nulos
Agora é hora de converter os valores nulos para que o modelo consiga interpretar todos os dados da tabela.

1. **Conversão categórica**: Já os nulos de colunas categóricas converteremos de para `-1` ou `IGNORADO` nos casos das colunas criadas pelo `pd.cut`
   - Isso vai facilitar na hora de fazermos o encoding

#### Por quê não fazer a conversão numérica agora?
 **Conversão numérica**: Os nulos de colunas numéricas converteremos para `mediana`, pois essa medida lida com outliers.
 Porém, essa etapa deve ser feita **SOMENTE após a separação da base de treino e base de teste**. Isso por quê quando ao trocarmos os dados por mediana ele irá considerar a mediana de 700k pessoas. Quando formos fazer o teste, o modelo já irá saber a média do banco de teste, provocando o **Data laeakage**.

 Sendo assim, seguiremos a seguinte ordem:
 1. iremos fazer a separação do banco entre treino e teste
 2. calculamos a mediana da base de treino
 3. aplicamos o valor da mediana da base de treino para a base de teste

In [11]:
numeric_cols = [
    'IDADEMAE', 'IDADEPAI','QTDFILVIVO', 'QTDFILMORT','CONSPRENAT', 'MESPRENAT',
    'QTDGESTANT','QTDPARTNOR','QTDPARTCES', 'LATITUDE', 'LONGITUDE',
]

categoric_cols = ['RACACORMAE', 'GRAVIDEZ', 'SEXO', 'FAIXAETAMAE', 'ESTCIVMAE', 'KOTELCHUCK', 'ESCMAE2010']

for col in categoric_cols:
    # Para colunas que viraram 'category' pelo pd.cut, precisamos adicionar o valor antes
    if col in df_clean.columns:
        if df_clean[col].dtype.name == 'category':
            df_clean[col] = df_clean[col].cat.add_categories('IGNORADO').fillna('IGNORADO')
        else:
            df_clean[col] = df_clean[col].fillna(-1)

# Verificação final de nulos
print(df_clean.isnull().sum().sum(), "nulos restantes.")

463043 nulos restantes.


#### Dicionário de Mapeamento (Humanização)

Algumas colunas do dataset já vem com valores numéricos, ao aplicar o `One-Hot Enconding` ele transforma o nome dessas colunas de acordo com os valores que encontra na mesma. O que dificulta a interpretabilidade, por isso, iremos "traduzir" os nomes das colunas utilizando o [dicionário disponibilizado](https://diaad.s3.sa-east-1.amazonaws.com/sinasc/SINASC+-+Estrutura.pdf) pelo `OpenDataSus`


In [12]:
# Selecionando as categorias que farã o One-Hot
one_hot_cols = categoric_cols.copy()

# Verificando os valores contidos nessas colunas
for col in one_hot_cols:
    print(f'{col} : {df_clean[col].unique()}')

RACACORMAE : ['1' '4' '2' '3' '5' -1]
GRAVIDEZ : [ 1.  2. -1.  3.]
SEXO : [ 1.  2. -1.]
FAIXAETAMAE : ['IDEAL', 'AVANCADA', 'ADOLESC', 'EXTRALTA', 'EXTBAIXA']
Categories (6, object): ['EXTBAIXA' < 'ADOLESC' < 'IDEAL' < 'AVANCADA' < 'EXTRALTA' < 'IGNORADO']
ESTCIVMAE : [ 1.  4.  2.  5. -1.  3.]
KOTELCHUCK : [ 5.  3.  2.  4.  1. -1.]
ESCMAE2010 : [ 5.  3.  2.  4.  1.  0. -1.]


In [13]:
# Converter colunas em inteiros 
one_hot_int_cols = one_hot_cols.copy()

# Removendo colunas que não precisarão da humanização
one_hot_int_cols.remove('FAIXAETAMAE')

# Transformando em inteiros
for col in one_hot_int_cols:
    if col in df_clean.columns:
        # errors='coerce' transforma sujeira em NaN, fillna(-1) garante o inteiro
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(-1).astype(int)

# Verificando a transformação
for col in one_hot_cols:
    print(f'{col} : {df_clean[col].unique()}')

RACACORMAE : [ 1  4  2  3  5 -1]
GRAVIDEZ : [ 1  2 -1  3]
SEXO : [ 1  2 -1]
FAIXAETAMAE : ['IDEAL', 'AVANCADA', 'ADOLESC', 'EXTRALTA', 'EXTBAIXA']
Categories (6, object): ['EXTBAIXA' < 'ADOLESC' < 'IDEAL' < 'AVANCADA' < 'EXTRALTA' < 'IGNORADO']
ESTCIVMAE : [ 1  4  2  5 -1  3]
KOTELCHUCK : [ 5  3  2  4  1 -1]
ESCMAE2010 : [ 5  3  2  4  1  0 -1]


In [14]:
sinasc_dictionary = {
    'SEXO': {1: 'MASC', 2: 'FEM', -1: 'IGNORADO'},
    'RACACORMAE': {1: 'BRANCA', 2: 'PRETA', 3: 'AMARELA', 4: 'PARDA', 5: 'INDIGENA', -1: 'IGNORADO'},
    'ESTCIVMAE': {1: 'SOLTEIRA', 2: 'CASADA', 3: 'VIUVA', 4: 'DIVORCIADA', 5: 'UNIAO_ESTAVEL', -1: 'IGNORADO'},
    'GRAVIDEZ': {1: 'UNICA', 2: 'DUPLA', 3: 'TRIPMAIS', -1: 'IGNORADO'},
    "KOTELCHUCK": {
        1: "SEM_PRENATAL",
        2: "INADEQUADO",
        3: "INTERMEDIARIO",
        4: "ADEQUADO",
        5: "MAIS_QUE_ADEQUADO",
        -1: "IGNORADO"
    },
    "ESCMAE2010": {
        0: "SEM_ESCOLARIDADE",
        1: "FUNDAMENTAL_I",
        2: "FUNDAMENTAL_II",
        3: "MEDIO",
        4: "SUPERIOR_INCOMPLETO", 
        5: "SUPERIOR_COMPLETO",
        -1: "IGNORADO"
    }
}

for col, value in sinasc_dictionary.items():
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].map(value).fillna('IGNORADO')     

df_clean

,SEMAGESTAC,GESTACAO,IDADEMAE,ESCMAE2010,ESTCIVMAE,RACACORMAE,QTDGESTANT,QTDPARTNOR,QTDPARTCES,QTDFILVIVO,QTDFILMORT,GRAVIDEZ,MESPRENAT,CONSPRENAT,SEXO,KOTELCHUCK,IDADEPAI,LATITUDE,LONGITUDE,PREMATURO,PAI_AUSENTE,IDADEPAI_INVALIDA,PNTARDIO,HISTPERDAFETAL,PRIMIPARA,FAIXAETAMAE
0,38.0,5.0,30,SUPERIOR_COMPLETO,SOLTEIRA,BRANCA,1.0,0.0,1.0,1.0,0.0,UNICA,1.0,11.0,MASC,MAIS_QUE_ADEQUADO,39.0,-19.9102,-43.9266,0,0,0,0,0,0,IDEAL
1,38.0,5.0,27,MEDIO,SOLTEIRA,BRANCA,0.0,0.0,0.0,0.0,0.0,UNICA,2.0,8.0,FEM,MAIS_QUE_ADEQUADO,39.0,-19.9102,-43.9266,0,0,0,0,0,1,IDEAL
2,40.0,5.0,41,SUPERIOR_COMPLETO,SOLTEIRA,PARDA,0.0,0.0,0.0,0.0,0.0,UNICA,1.0,9.0,FEM,MAIS_QUE_ADEQUADO,36.0,-19.9102,-43.9266,0,0,0,0,0,1,AVANCADA
3,37.0,5.0,29,MEDIO,SOLTEIRA,BRANCA,0.0,0.0,0.0,0.0,0.0,UNICA,1.0,9.0,FEM,MAIS_QUE_ADEQUADO,22.0,-19.9102,-43.9266,0,0,0,0,0,1,IDEAL
4,40.0,5.0,36,SUPERIOR_COMPLETO,DIVORCIADA,BRANCA,2.0,0.0,0.0,0.0,2.0,UNICA,1.0,9.0,FEM,MAIS_QUE_ADEQUADO,38.0,-19.9102,-43.9266,0,0,0,0,1,0,AVANCADA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
717927,37.0,5.0,37,MEDIO,CASADA,PARDA,2.0,0.0,1.0,1.0,1.0,UNICA,2.0,11.0,MASC,MAIS_QUE_ADEQUADO,38.0,-21.3009,-46.7964,0,0,0,0,1,0,AVANCADA
717928,39.0,5.0,37,MEDIO,CASADA,PARDA,1.0,0.0,1.0,1.0,0.0,UNICA,2.0,8.0,MASC,MAIS_QUE_ADEQUADO,31.0,-21.3050,-46.7081,0,0,0,0,0,0,AVANCADA
717929,40.0,5.0,32,FUNDAMENTAL_II,SOLTEIRA,IGNORADO,1.0,0.0,1.0,1.0,0.0,UNICA,1.0,12.0,FEM,MAIS_QUE_ADEQUADO,22.0,-21.3590,-46.9401,0,0,0,0,0,0,IDEAL
717930,41.0,5.0,15,MEDIO,SOLTEIRA,IGNORADO,0.0,0.0,0.0,0.0,0.0,UNICA,1.0,7.0,FEM,MAIS_QUE_ADEQUADO,19.0,-21.3050,-46.7081,0,0,0,0,0,1,ADOLESC


### Preparação e Treino (Split)

Nesta etapa, pegamos o seu dataset original e o dividimos em dois grupos distintos: **Treino** (geralmente 80%) e **Teste** (20%).

- **O que acontece:** O grupo de treino é o "livro" que o modelo vai ler para aprender. O grupo de teste é a "prova final" com questões que ele nunca viu antes.
- **A "Estratificação":** Como você tem poucos prematuros (11%), usamos o `stratify=y` para garantir que tanto o treino quanto o teste tenham exatamente esses mesmos 11%. Sem isso, você poderia acabar com um teste sem nenhum prematuro para validar!


In [15]:
from sklearn.model_selection import train_test_split

In [16]:
# 1. Definindo o Target (y)
y = df_clean['PREMATURO']

# 2. Definindo as Features (X)
# IMPORTANTE: Removemos colunas que "vazam" a resposta (Target Leakage)
# 'CONSPRENAT' é removida porque bebês prematuros têm menos consultas por falta de tempo, 
# e não necessariamente por falta de assistência prévia.
cols_to_drop = ['SEMAGESTAC', 'GESTACAO', 'PREMATURO', 'CONSPRENAT']
X = df_clean.drop(columns=[c for c in cols_to_drop if c in df_clean.columns])

# 3. Split Estratificado (80% treino, 20% teste)
# O stratify=y é para manter a proporção de prematuros em ambos os sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [17]:
y_train.value_counts()

PREMATURO
0    500246
1     62274
Name: count, dtype: int64

In [18]:
# Filtrando por colunas existentes em X_train
numeric_cols_model = [col for col in numeric_cols if col in X_train.columns]

# Aplicação da imputação nas bases como citado na etapa de Imputação
for col in numeric_cols_model:
    # Calculamos a mediana APENAS no treino
    train_median = X_train[col].median()
    
    # Aplicamos a mediana do treino em ambos
    X_train[col] = X_train[col].fillna(train_median)
    X_test[col] = X_test[col].fillna(train_median)


## Enconding
Nesta etapa, convertemos colunas categóricas em numéricas para que o modelo possa processá-las matematicamente, seguindo dois critérios:

- **Diferenciação de Pesos**: Para variáveis sem hierarquia (como `SEXO` ou `RACACORMAE`), utilizamos o **One-Hot Encoding**. Isso evita que o modelo atribua pesos errados (ex: interpretar que **`2 > 1`**) a categorias puramente nominais.
- **Estratégia de Integridade (Anti-Leakage)**: Realizamos o cálculo de imputação (como a **mediana**) apenas após a separação entre treino e teste.

#### Por que separar antes?
Para garantir que a estatística do conjunto de teste não "vaze" para o treino. Em produção, o modelo usará a "régua" aprendida no passado (treino) para lidar com novos dados, garantindo uma avaliação real de sua performance.

In [19]:
# Criar as dummies
X_train_encoded = pd.get_dummies(X_train, columns=one_hot_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=one_hot_cols, drop_first=True)


# O reindex garante que o X_test tenha EXATAMENTE as mesmas colunas do X_train.
# Se uma coluna caiu no drop do treino, ela cai no teste. # Se uma categoria não existia no teste, ele cria a coluna com 0.
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

# Converte tudo que for booleano para int (Garante o tipo)
bool_cols = X_train_encoded.select_dtypes(include=['bool']).columns
X_train_encoded[bool_cols] = X_train_encoded[bool_cols].astype(int)
X_test_encoded[bool_cols] = X_test_encoded[bool_cols].astype(int)


In [20]:
X_train_encoded.head(10)

,IDADEMAE,QTDGESTANT,QTDPARTNOR,QTDPARTCES,QTDFILVIVO,QTDFILMORT,MESPRENAT,IDADEPAI,LATITUDE,LONGITUDE,PAI_AUSENTE,IDADEPAI_INVALIDA,PNTARDIO,HISTPERDAFETAL,PRIMIPARA,RACACORMAE_BRANCA,RACACORMAE_IGNORADO,RACACORMAE_INDIGENA,RACACORMAE_PARDA,RACACORMAE_PRETA,GRAVIDEZ_IGNORADO,GRAVIDEZ_TRIPMAIS,GRAVIDEZ_UNICA,SEXO_IGNORADO,SEXO_MASC,FAIXAETAMAE_ADOLESC,FAIXAETAMAE_IDEAL,FAIXAETAMAE_AVANCADA,FAIXAETAMAE_EXTRALTA,FAIXAETAMAE_IGNORADO,ESTCIVMAE_DIVORCIADA,ESTCIVMAE_IGNORADO,ESTCIVMAE_SOLTEIRA,ESTCIVMAE_UNIAO_ESTAVEL,ESTCIVMAE_VIUVA,KOTELCHUCK_IGNORADO,KOTELCHUCK_INADEQUADO,KOTELCHUCK_INTERMEDIARIO,KOTELCHUCK_MAIS_QUE_ADEQUADO,KOTELCHUCK_SEM_PRENATAL,ESCMAE2010_FUNDAMENTAL_II,ESCMAE2010_IGNORADO,ESCMAE2010_MEDIO,ESCMAE2010_SEM_ESCOLARIDADE,ESCMAE2010_SUPERIOR_COMPLETO,ESCMAE2010_SUPERIOR_INCOMPLETO
359929,30,2.0,0.0,1.0,1.0,1.0,2.0,31.0,-20.3933,-42.1533,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0
424664,25,0.0,0.0,0.0,0.0,0.0,2.0,32.0,-15.3168,-42.0213,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0
361636,25,0.0,0.0,0.0,0.0,0.0,3.0,32.0,-17.6973,-40.7723,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0
697249,19,0.0,0.0,0.0,0.0,0.0,2.0,32.0,-19.9668,-44.2008,1,0,-1,0,1,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0
651664,32,1.0,1.0,0.0,1.0,0.0,2.0,32.0,-18.5712,-41.2340,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0
689735,21,0.0,0.0,0.0,0.0,0.0,2.0,32.0,-19.9102,-43.9266,1,0,0,0,1,0,0,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0
456586,24,1.0,1.0,0.0,1.0,0.0,4.0,23.0,-19.5210,-44.4544,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1
612870,29,0.0,0.0,0.0,0.0,0.0,1.0,32.0,-19.9102,-43.9266,1,0,0,0,1,0,0,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0
307760,33,1.0,0.0,1.0,1.0,0.0,1.0,38.0,-19.7621,-44.0844,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0
564963,30,2.0,2.0,0.0,2.0,0.0,6.0,32.0,-16.8523,-42.0637,1,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0


In [21]:
X_test_encoded.head(10)

,IDADEMAE,QTDGESTANT,QTDPARTNOR,QTDPARTCES,QTDFILVIVO,QTDFILMORT,MESPRENAT,IDADEPAI,LATITUDE,LONGITUDE,PAI_AUSENTE,IDADEPAI_INVALIDA,PNTARDIO,HISTPERDAFETAL,PRIMIPARA,RACACORMAE_BRANCA,RACACORMAE_IGNORADO,RACACORMAE_INDIGENA,RACACORMAE_PARDA,RACACORMAE_PRETA,GRAVIDEZ_IGNORADO,GRAVIDEZ_TRIPMAIS,GRAVIDEZ_UNICA,SEXO_IGNORADO,SEXO_MASC,FAIXAETAMAE_ADOLESC,FAIXAETAMAE_IDEAL,FAIXAETAMAE_AVANCADA,FAIXAETAMAE_EXTRALTA,FAIXAETAMAE_IGNORADO,ESTCIVMAE_DIVORCIADA,ESTCIVMAE_IGNORADO,ESTCIVMAE_SOLTEIRA,ESTCIVMAE_UNIAO_ESTAVEL,ESTCIVMAE_VIUVA,KOTELCHUCK_IGNORADO,KOTELCHUCK_INADEQUADO,KOTELCHUCK_INTERMEDIARIO,KOTELCHUCK_MAIS_QUE_ADEQUADO,KOTELCHUCK_SEM_PRENATAL,ESCMAE2010_FUNDAMENTAL_II,ESCMAE2010_IGNORADO,ESCMAE2010_MEDIO,ESCMAE2010_SEM_ESCOLARIDADE,ESCMAE2010_SUPERIOR_COMPLETO,ESCMAE2010_SUPERIOR_INCOMPLETO
551228,40,3.0,1.0,1.0,2.0,1.0,3.0,32.0,-20.1446,-44.8912,1,0,0,1,0,1,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0
51442,21,1.0,1.0,0.0,1.0,0.0,1.0,24.0,-19.7472,-47.9381,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0
245630,27,1.0,1.0,0.0,1.0,0.0,2.0,32.0,-21.7595,-43.3398,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0
506848,24,0.0,0.0,0.0,0.0,0.0,2.0,38.0,-18.5699,-46.5013,0,0,0,0,1,0,0,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
661642,34,4.0,3.0,1.0,4.0,0.0,8.0,32.0,-21.7595,-43.3398,1,0,1,0,0,0,0,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0
203720,21,0.0,0.0,0.0,0.0,0.0,1.0,28.0,-19.9321,-44.0539,0,0,0,0,1,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0
287140,29,0.0,0.0,0.0,0.0,0.0,1.0,33.0,-18.9141,-48.2749,0,0,0,0,1,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0
703013,30,0.0,0.0,0.0,0.0,0.0,2.0,30.0,-20.6134,-42.1438,0,0,0,0,1,1,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0
496505,30,1.0,0.0,1.0,1.0,0.0,3.0,35.0,-19.5811,-42.6471,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0
386597,32,3.0,1.0,0.0,1.0,2.0,2.0,32.0,-21.0927,-45.5612,1,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1


In [22]:
X_train_encoded.shape

(562520, 46)

#### A quantidade de colunas AUMENTOU e agora? 

Será que mesmo que a quantidade de colunas tenha aumentado vale a pena considerar uma prática de dimensionamento? Por quê? 

A resposta é **não**.

##### Motivos:
- Contém colunas essenciais
- O tamanho do dataset é considerado saudável para modelos atuais.
  - A diferença de 21 para 46 colunas para modelos atuais é irrelevante pois mesmo que tenha dobrado, ainda temos apenas 700k de linhas. O que é uma quantidade boa para faze um modelo legal, mas não é uma quantidade absurda.
- Se aplicarmos um **PCA** aqui iriamos perder a relevância clínica (que é a ALMA do nosso dataset)


### Padronização (Scaling)

Aqui transformamos todas as variáveis para que tenham a mesma escala (média 0 e desvio padrão 1).

- **O que acontece:** A **Latitude** (ex: -23) e o **Número de Consultas** (ex: 7) são números de escalas muito diferentes. O Scaler coloca todos em uma "régua" comum.
- **O detalhe técnico:** O modelo para de olhar para o valor absoluto e passa a olhar para o quanto aquele valor se desvia da média.

In [23]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_encoded), columns=X_train_encoded.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_encoded), columns=X_test_encoded.columns)

#### Por que fazemos nessa ordem?
A ordem correta é **Split primeiro, Padronização depois**. O motivo é evitar o **Data Leakage** (Vazamento de Dados).

Se padronizarmos o dataset inteiro **antes** de dividir, o cálculo da média e do desvio padrão incluiria informações dos dados de teste. Isso significa que:

1. O grupo de treino "saberia" informações sobre a distribuição global dos dados que ele não deveria saber.
2. Isso cria um modelo "viciado" que performa muito bem no seu notebook, mas falha miseravelmente quando recebe dados reais de um hospital no futuro, pois ele aprendeu com base em médias que incluíam o "futuro" (o teste).

### Persistência dos artefatos

Salva `X_train_scaled`, `X_test_scaled`, `y_train` e `y_test` em `data/` para que o `03_modeling.ipynb` possa carregá-los sem re-executar todo o pipeline.

O índice de `y_train`/`y_test` é resetado para alinhar com o índice 0-based criado pelo `StandardScaler` no passo anterior.

In [24]:
from utils import save_if_different

y_train_reset = y_train.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

save_if_different(X_train_scaled, '../data/X_train.parquet')
save_if_different(X_test_scaled, '../data/X_test.parquet')
save_if_different(y_train_reset.to_frame(), '../data/y_train.parquet')
save_if_different(y_test_reset.to_frame(), '../data/y_test.parquet')

Arquivo salvo com sucesso
Arquivo salvo com sucesso
Arquivo salvo com sucesso
Arquivo salvo com sucesso
